In [2]:
# Load only from python 3.12
import sys

sys.path = [p for p in sys.path if "python3.8" not in p]

In [2]:
# !{sys.executable} -m pip install osmium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 4.3 MB/s  0:00:00m eta 0:00:01


In [3]:
# Import
import os
import osmium
import geopandas as gpd
import pandas as pd
import rasterio
import requests
import numpy as np
from shapely.geometry import box
from concurrent.futures import ProcessPoolExecutor, as_completed

In [4]:
# Read osm link data
osm_link = pd.read_csv("data/osm_link.csv")

In [13]:
# Configure directiry and tags
DOWNLOAD_DIR = "data/osm_raw"
FILTERED_DIR = "data/osm_filtered"
MAX_WORKERS = os.cpu_count() - 1 or 1

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(FILTERED_DIR, exist_ok=True)

TARGET_PARKING_VALUES = {
    "surface", "street_side", "lane",
    "on_kerb", "half_on_kerb", "shoulder", "layby"
}

In [8]:
# Download
def download_file(url, out_path):
    if os.path.exists(out_path):
        return out_path

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
    return out_path

In [9]:
# Filter with pyosmium
class ParkingFilterHandler(osmium.SimpleHandler):
    def __init__(self, writer):
        super().__init__()
        self.writer = writer

    def _has_parking(self, tags):
        has_amenity_parking = False
        has_valid_parking_type = False
        for k, v in tags:
            if k == "amenity" and v == "parking":
                has_amenity_parking = True

            elif k.startswith("parking:"):
                has_valid_parking_type = True

        return has_amenity_parking or has_valid_parking_type

    def node(self, n):
        if self._has_parking(n.tags):
            self.writer.add_node(n)

    def way(self, w):
        if self._has_parking(w.tags):
            self.writer.add_way(w)

    def relation(self, r):
        if self._has_parking(r.tags):
            self.writer.add_relation(r)


def filter_pbf(input_pbf, output_pbf):
    writer = osmium.SimpleWriter(output_pbf)
    handler = ParkingFilterHandler(writer)
    handler.apply_file(input_pbf, locations=True)
    writer.close()

In [10]:
# Workflow
def process_region(row):
    region = row["Region"]
    url = row["Links"]

    filename = url.split("/")[-1]
    raw_path = os.path.join(DOWNLOAD_DIR, filename)
    filtered_path = os.path.join(FILTERED_DIR, f"{region}_parking.osm.pbf")

    try:
        # Download
        # download_file(url, raw_path)

        # Filter
        filter_pbf(raw_path, filtered_path)

        return (region, "OK", filtered_path)

    except Exception as e:
        return (region, "ERROR", str(e))

In [11]:
# Run query
def run_parallel(df):
    results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(process_region, row)
            for row in df.to_dict("records")
        ]

        for future in as_completed(futures):
            result = future.result()
            region, status, info = result

            if status == "OK":
                print(f"[OK] {region}")
            else:
                print(f"[FAIL] {region} → {info}")

            results.append(result)

    return results

In [ ]:
filtered_osm = run_parallel(osm_link)

[OK] Monaco
[OK] Liechtenstein
[OK] Guernsey and Jersey
[OK] Andorra
[OK] Isle of Man
[OK] Malta
[OK] Faroe Islands
[OK] Azores
[OK] Macedonia
[OK] Kosovo
[OK] Montenegro
[OK] Luxembourg
[OK] Cyprus
[OK] Albania
[OK] Iceland
[OK] Georgia
[OK] Moldova
[OK] Estonia
[OK] Latvia
[OK] Lithuania
[OK] Bulgaria
[OK] Bosnia-Herzegovina
[OK] Croatia
[OK] Serbia
[OK] Hungary
[OK] Slovakia
[OK] Romania
[OK] Ireland and Northern Ireland
[OK] Slovenia
[OK] Switzerland
[OK] Greece
[OK] Belarus
[OK] Denmark
[OK] Portugal
[OK] Austria
[OK] Belgium
[OK] Finland
[OK] Turkey
[OK] Czech Republic
[OK] Sweden
[OK] Netherlands
[OK] Spain
[OK] Norway
[OK] Poland
[OK] United Kingdom
[OK] Italy
[OK] Germany


In [18]:
filtered_osm

[('Monaco', 'OK', 'data/osm_filtered/Monaco_parking.osm.pbf'),
 ('Liechtenstein', 'OK', 'data/osm_filtered/Liechtenstein_parking.osm.pbf'),
 ('Guernsey and Jersey',
  'OK',
  'data/osm_filtered/Guernsey and Jersey_parking.osm.pbf'),
 ('Andorra', 'OK', 'data/osm_filtered/Andorra_parking.osm.pbf'),
 ('Isle of Man', 'OK', 'data/osm_filtered/Isle of Man_parking.osm.pbf'),
 ('Malta', 'OK', 'data/osm_filtered/Malta_parking.osm.pbf'),
 ('Faroe Islands', 'OK', 'data/osm_filtered/Faroe Islands_parking.osm.pbf'),
 ('Azores', 'OK', 'data/osm_filtered/Azores_parking.osm.pbf'),
 ('Macedonia', 'OK', 'data/osm_filtered/Macedonia_parking.osm.pbf'),
 ('Kosovo', 'OK', 'data/osm_filtered/Kosovo_parking.osm.pbf'),
 ('Montenegro', 'OK', 'data/osm_filtered/Montenegro_parking.osm.pbf'),
 ('Luxembourg', 'OK', 'data/osm_filtered/Luxembourg_parking.osm.pbf'),
 ('Cyprus', 'OK', 'data/osm_filtered/Cyprus_parking.osm.pbf'),
 ('Albania', 'OK', 'data/osm_filtered/Albania_parking.osm.pbf'),
 ('Iceland', 'OK', 'data/o

## Create grid

In [ ]:
# Config
GRID_SIZE = 500  # meters
CRS = "EPSG:3035"

# Approx Europe extent in EPSG:3035
xmin, ymin = 2500000, 1300000
xmax, ymax = 6500000, 5500000

In [ ]:
# Built function
def create_grid(xmin, xmax, ymin, ymax, crs, grid_size=500):
    xs = np.arange(xmin, xmax, grid_size)
    ys = np.arange(ymin, ymax, grid_size)
    
    grid = gpd.GeoDataFrame(
        geometry=[box(x, y, x+grid_size, y+grid_size) for x in xs for y in ys],
        crs=crs
    )
    
    grid.to_parquet("data/eu_grid_500m.parquet")
    return grid

In [ ]:
eu_grid = create_grid(xmin, xmax, ymin, ymax, CRS, GRID_SIZE)